# Colab — AIHub 데이터셋 학습 → Drive 저장

> 대상 단계: **③ 위험요소 탐지** · 클래스 `0 person` · `1 stairs` · `2 bollard` (8/3 확정)
> 근거 문서: [`docs/data.md` 3-1](../docs/data.md) · [`docs/detection.md` 8-3](../docs/detection.md) · [`docs/STATUS.md`](../docs/STATUS.md)

## ⚠️ 먼저 — 이 노트북은 **다운로드를 하지 않는다**

처음엔 Colab 에서 `aihubshell` 로 직접 받으려 했는데 **막힌다** (2026-08-04 실측):

```
Download failed with HTTP status 502.
AI 허브는 해외에서의 데이터 다운로드를 제한하고 있습니다.
```

Colab 런타임은 미국·유럽에 뜬다. 우회는 AIHub 이용정책 위반이므로 하지 않는다.
그래서 역할을 나눴다.

```
국내 PC   AIHub 다운로드 → 압축해제 → scripts/aihub_pack_for_colab.py
              → YOLO 변환 + 640 리사이즈 + zip  (30GB → 2~3GB)
                 ↓  Drive 업로드
Colab     이 노트북 — 학습 → best.pt → Drive
                 ↓
국내 PC   eval_nightowls.py (야간 판정) · export_onnx.py (앱 전달물)
```

**640 으로 줄이는 것은 손실이 아니다.** 학습 입력이 어차피 `imgsz=640` 이라 ultralytics 가
같은 크기로 줄인다. 미리 줄여 두면 옮길 수 있는 크기가 되고 학습 I/O 도 가벼워진다.

### 국내 PC 에서 먼저 할 것

```powershell
# 1) AIHub 에서 zip 다운로드 (웹 or aihubshell — 국내 IP 여야 한다)
#    바운딩박스 1개(10GB, person·bollard) + 서피스마스킹 1개(10GB, stairs) 로 정찰
#    ⚠️ 폴리곤세그멘테이션·뎁스는 받지 않는다 (전자는 같은 장애물 29종, 후자는 무관)
# 2) 압축 해제 후
uv run python scripts/aihub_pack_for_colab.py --src D:\datasets\AIHub인도보행 --dry-run  # 분포만
uv run python scripts/aihub_pack_for_colab.py --src D:\datasets\AIHub인도보행 --zip
# 3) outputs/datasets/aihub_colab.zip 을 Drive 의 bammasil/datasets/ 에 업로드
```

## ⚠️ 실행 환경 — 이 노트북은 `uv` 와 무관하다

**`pyproject.toml` · `uv.lock` 을 건드리지 않는다.** 여기서 설치하는 것은 Colab 런타임
안에서만 살아 있고 세션이 끝나면 사라진다. 로컬은 지금까지처럼 `uv run` 그대로다.

### VS Code Colab 확장으로 열어도 되지만 — 학습은 웹 Colab 을 권한다

셀은 런타임(구글 서버)에서 도니 결과는 같다. 다만 웹 UI 에 의존하는 기능 2개가 안 된다.

| 기능 | 웹 Colab | VS Code |
|---|---|---|
| 🔑 보안 비밀 (`userdata`) | 됨 | ❌ 패널이 없다 |
| `drive.mount()` | 됨 | ❌ `[dfs_ephemeral] Credentials propagation unsuccessful` |

두 번째가 치명적이다 — Drive 가 없으면 **데이터셋을 가져올 수도, 학습 결과를 내보낼
수도 없다** (파일 패널·`files.download()` 도 전부 웹 UI 기능이다).
**이 노트북은 [colab.research.google.com](https://colab.research.google.com) 에서
실행한다.** 편집은 VS Code 에서 계속하면 된다 — 같은 파일이다.

## ⚠️ 알고 있어야 하는 함정 3가지

프로젝트가 이미 대가를 치르고 배운 것들이다 (→ [STATUS 3장](../docs/STATUS.md)).

1. **개발 val 로 판정하지 말 것.** `C4b` 에서 실야간 성능이 5.4배 좋아지는 동안
   개발 val 은 0.892 → 0.892 로 한 자리도 안 움직였다. 판정은 held-out(NightOwls
   rec 34) 또는 자체 촬영분(`C5`)에서 한다 — **둘 다 로컬에 있다.**
2. **AIHub 는 주간 전용이다.** 야간 성능은 여기서 판정되지 않는다.
3. **분할은 이미 블록 단위로 되어 있다.** 연속 영상 프레임이라 무작위로 나누면
   train/val 에 쌍둥이가 갈려 val 이 부풀어 오른다. 여기서 다시 섞지 말 것.

---
## 1. 런타임 확인

GPU 가 안 잡혀 있으면 **런타임 → 런타임 유형 변경 → T4 GPU**.

In [17]:
import shutil, subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()
print("GPU  :", gpu or "⚠️ 없음 — 런타임 유형을 GPU 로 바꿀 것")

total, used, free = shutil.disk_usage("/content")
print(f"디스크: 전체 {total / 2**30:.0f}GiB · 여유 {free / 2**30:.0f}GiB")

GPU  : Tesla T4, 15360 MiB, 580.82.07
디스크: 전체 113GiB · 여유 66GiB


---
## 2. 설치 — **Colab 런타임 안에서만**

`torch` 는 Colab 에 이미 CUDA 빌드로 있다. **다시 설치하지 않는다** — 건드리면 런타임의
CUDA 스택이 어긋난다. 추가하는 것은 `ultralytics` 하나뿐이다.

In [18]:
%pip install -q "ultralytics>=8.3.0"

import torch, ultralytics

print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "· CUDA", torch.cuda.is_available(),
      "·", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print()
print("↑ Colab 세션 전용이다. pyproject.toml / uv.lock 에 반영하지 말 것.")

ultralytics 8.4.115
torch       2.11.0+cu128 · CUDA True · Tesla T4

↑ Colab 세션 전용이다. pyproject.toml / uv.lock 에 반영하지 말 것.


---
## 3. 리포 clone

클래스 id 의 유일한 정의처(`scripts/nightowls_yolo.CLASS_NAMES`)를 임포트해 **데이터셋의
클래스 배치가 프로젝트와 같은지** 확인하는 용도다. 노트북에 id 를 다시 적으면
소스별로 어긋난다.

In [19]:
import sys
from pathlib import Path

REPO   = "https://github.com/kty2001/KDT_Hackathon.git"
BRANCH = "main"
REPO_DIR = Path("/content/KDT_Hackathon")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO} {REPO_DIR}

sys.path.insert(0, str(REPO_DIR / "scripts"))
from nightowls_yolo import CLASS_NAMES

print("\n클래스 (단일 정의처):", CLASS_NAMES)

Already up to date.

클래스 (단일 정의처): {0: 'person', 1: 'stairs', 2: 'bollard'}


---
## 4. Drive 마운트

| 경로 | 내용 |
|---|---|
| `Drive/bammasil/datasets/aihub_colab.zip` | ★ **국내에서 만들어 올린 데이터셋** (입력) |
| `Drive/bammasil/runs/` | 학습 결과·가중치 (출력) |
| `/content/work/` | 압축 해제본·학습 중간 산출물 (⚠️ 세션 종료 시 소멸) |

> 마운트가 실패하면 **VS Code 로 붙은 것**이다. 웹 Colab 에서 열 것 (0장 참고).

In [20]:
WORK = Path("/content/work")
RUNS = WORK / "runs"

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/bammasil")
for p in (WORK, RUNS, DRIVE / "runs", DRIVE / "datasets"):
    p.mkdir(parents=True, exist_ok=True)

print("작업 루트:", WORK)
print("Drive    :", DRIVE)
!ls -lh {DRIVE}/datasets/ 2>/dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
작업 루트: /content/work
Drive    : /content/drive/MyDrive/bammasil
total 19M
drwx------ 3 root root 4.0K Aug  4 08:30 aihub_colab_rehearsal
-rw------- 1 root root  19M Aug  4 08:20 aihub_colab_rehearsal.zip


---
## 5. 데이터셋 가져오기

Drive 에 올려 둔 zip 을 **런타임 로컬로** 푼다. Drive 에 둔 채로 학습하면 안 된다 —
FUSE 마운트라 소파일 수만 개를 읽는 학습 I/O 에서 GPU 가 놀게 된다.

> ⚠️ **`data.yaml` 의 `path:` 를 여기서 다시 쓴다.** ultralytics 는 `path:` 가
> **상대경로면 `settings['datasets_dir']`**(Colab 은 `/content/datasets`) 기준으로 풀어서,
> 패키징한 머신의 경로가 박혀 있으면
> `images not found, missing path '/content/datasets/outputs/...'` 로 죽는다 (8/4 실측).
> 압축을 푼 **실제 위치**로 덮어쓰면 어떤 zip 이든 돌아간다.

In [ ]:
# 본 데이터: aihub_colab.zip · 리허설(샘플 19MB): aihub_colab_rehearsal.zip
DATASET_ZIP = DRIVE / "datasets/aihub_colab.zip"

assert DATASET_ZIP.is_file(), (
    f"{DATASET_ZIP} 이 없다.\n"
    "국내 PC 에서 먼저:\n"
    "  uv run python scripts/aihub_pack_for_colab.py --src <다운로드분> --zip\n"
    "그리고 outputs/datasets/aihub_colab.zip 을 Drive 의 bammasil/datasets/ 에 업로드")

DATA_DIR = WORK / "datasets" / DATASET_ZIP.stem
DATA_DIR.mkdir(parents=True, exist_ok=True)
!unzip -q -o {DATASET_ZIP} -d {DATA_DIR}

# zip 이 한 겹 더 싸여 있을 수도 있다 — data.yaml 을 실제로 찾아서 쓴다
found = sorted(DATA_DIR.rglob("data.yaml"), key=lambda p: len(p.parts))
assert found, f"{DATA_DIR} 안에 data.yaml 이 없다 — zip 을 잘못 만들었다"
DATA_YAML = found[0]
DATA_DIR = DATA_YAML.parent

# ★ path: 를 압축 푼 실제 위치로 덮어쓴다 (위 경고 참고)
import re
_t = DATA_YAML.read_text(encoding="utf-8")
_t = (re.sub(r"(?m)^path:.*$", f"path: {DATA_DIR.as_posix()}", _t)
      if re.search(r"(?m)^path:", _t) else f"path: {DATA_DIR.as_posix()}\n" + _t)
DATA_YAML.write_text(_t, encoding="utf-8")

print(DATA_YAML, "\n")
print(_t)

/content/work/datasets/aihub_colab_rehearsal/data.yaml 

# scripts/aihub_pack_for_colab.py 가 생성한다. 직접 수정하지 말 것.
# AIHub 인도보행 · 긴 변 640px · ⚠️ 주간 전용
path: /content/work/datasets/aihub_colab_rehearsal
train: images/train
val: images/val
names:
  0: person
  1: stairs
  2: bollard



### 5-1. 받은 것이 맞는지 확인한다

zip 안의 `stats.json` 은 **국내에서 잰 분포 실측 결과**다. 여기서 두 가지를 본다.

- `bollard` 박스 수 — 샘플에서는 **172개뿐**이었다(NightOwls person 은 7,972개).
  이게 여전히 적으면 zip 을 더 받아 다시 패키징하는 편이 낫다.
- `노면_stairs.학습투입` — StairNet 대조 게이트의 결과. `false` 면 노면 계단은
  **평가 전용**으로 빠져 있다 (→ [detection.md 8-3](../docs/detection.md)).

In [24]:
import json
from collections import Counter

stats_path = DATA_DIR / "stats.json"
if stats_path.is_file():
    stats = json.loads(stats_path.read_text(encoding="utf-8"))
    print(json.dumps(stats, ensure_ascii=False, indent=2)[:2000])
else:
    stats = {}
    print("⚠️ stats.json 이 없다 — 구버전 zip 이다. 아래 집계로 대신한다.")

print("\n" + "=" * 60)
counts = {}
for split in ("train", "val"):
    img_dir, lab_dir = DATA_DIR / "images" / split, DATA_DIR / "labels" / split
    n_img = len(list(img_dir.iterdir())) if img_dir.is_dir() else 0
    per = Counter()
    for f in lab_dir.glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                per[int(line.split()[0])] += 1
    counts[split] = n_img
    detail = " · ".join(f"{CLASS_NAMES[c]} {per[c]:,}" for c in sorted(per)) or "박스 없음"
    print(f"[{split}] 이미지 {n_img:,}장 · {detail}")

assert counts.get("train"), "train 이미지가 0 이다 — zip 구조를 확인할 것"

{
  "생성": "scripts/aihub_pack_for_colab.py",
  "원본": "data\\AIHub인도보행영상_sample",
  "클래스": {
    "0": "person",
    "1": "stairs",
    "2": "bollard"
  },
  "imgsz": 640,
  "장애물": {
    "프레임": 233,
    "person": 525,
    "bollard": 175,
    "버린_클래스": {
      "car": 773,
      "pole": 393,
      "tree_trunk": 321,
      "traffic_sign": 211,
      "potted_plant": 190,
      "movable_signage": 173,
      "truck": 162,
      "traffic_light": 161,
      "motorcycle": 118,
      "bicycle": 96
    }
  },
  "노면_stairs": {
    "프레임": 0,
    "박스": 0,
    "학습투입": false,
    "게이트": {}
  },
  "분할": {
    "train": {
      "이미지": 185,
      "박스": 565
    },
    "val": {
      "이미지": 45,
      "박스": 130
    }
  }
}

[train] 이미지 185장 · person 430 · bollard 135
[val] 이미지 45장 · person 93 · bollard 37


---
## 6. 학습

하이퍼파라미터를 `scripts/train_detect.py` 와 **똑같이** 맞춘다. 다르게 두면 로컬 런과
비교가 성립하지 않는다.

- 야간 도메인이라 색상 증강은 보수적 (`hsv_h=0.010 · hsv_s=0.4 · hsv_v=0.3`)
- 계단·보행자 모두 상하 반전이 물리적으로 무의미 (`flipud=0.0`)
- 배포 타깃과 맞춘 **YOLO11n**

> ★ **`project` 를 Drive 로 둔다.** 세션이 12시간·유휴 90분에 끊기는데, 런타임 안에
> 쓰면 그때까지의 체크포인트가 통째로 날아간다. Drive 에 쓰면 `last.pt` 가 남아
> **이어서 학습**할 수 있다. 대신 epoch 마다 쓰기가 조금 느려지는데, 몇 시간짜리
> 학습에서 이 보험이 훨씬 싸다.

In [25]:
from ultralytics import YOLO

RUN_NAME = "c4c_aihub_colab"
EPOCHS   = 100
BATCH    = 32     # T4 16GB 기준. OOM 이면 16 으로

PROJECT = DRIVE / "runs"          # ★ 끊겨도 남도록 Drive 에 직접 쓴다
RESUME  = (PROJECT / RUN_NAME / "weights/last.pt").is_file()
if RESUME:
    print("★ last.pt 가 있다 — 중단된 학습을 이어서 진행한다")

model = YOLO(str(PROJECT / RUN_NAME / "weights/last.pt") if RESUME else "yolo11n.pt")
model.train(
    data=str(DATA_YAML),
    resume=RESUME,
    epochs=EPOCHS,
    patience=20,
    imgsz=640,
    batch=BATCH,
    workers=2,
    cache=False,
    device=0,
    seed=0,
    deterministic=True,
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    hsv_h=0.010, hsv_s=0.4, hsv_v=0.3,
    flipud=0.0, fliplr=0.5,
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/datasets/aihub_colab_rehearsal/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=c4c_aih

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a41e6697500>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

---
## 7. 클래스별 성능

`bollard` 의 recall 을 특히 본다. **낮게 나오는 것이 예상된 결과다** — 640 환산 폭
중앙이 7.8px 이고 **50.3% 가 8px 미만**인데 YOLO11 의 최소 stride 가 8 이다. 절반이
특징맵 한 칸도 못 채운다. 데이터를 더 넣어 풀리는 문제가 아니라 입력 해상도의 구조적
한계다 (→ [data.md 3-1-1](../docs/data.md)).

**그래서 이 숫자는 "실패"가 아니라 "처음 재 본 값"이다** — 예상은 했지만 잰 적이 없다.

In [26]:
metrics = model.val(data=str(DATA_YAML), imgsz=640, device=0,
                    project=str(PROJECT), name=f"{RUN_NAME}_val", exist_ok=True)

names = model.names
print(f"{'class':<10}{'mAP50':>10}{'mAP50-95':>12}{'precision':>12}{'recall':>10}")
print("-" * 54)
per_class = {}
for i, c in enumerate(metrics.box.ap_class_index):
    row = dict(mAP50=float(metrics.box.ap50[i]), mAP=float(metrics.box.ap[i]),
               precision=float(metrics.box.p[i]), recall=float(metrics.box.r[i]))
    per_class[names[int(c)]] = row
    print(f"{names[int(c)]:<10}{row['mAP50']:>10.3f}{row['mAP']:>12.3f}"
          f"{row['precision']:>12.3f}{row['recall']:>10.3f}")
print(f"{'(전체)':<10}{metrics.box.map50:>10.3f}{metrics.box.map:>12.3f}")

print("\n" + "=" * 70)
print("⚠️ 이 val 은 **개발용**이다. 판정으로 쓰지 말 것.")
print("   `C4b` 에서 실야간이 5.4배 좋아지는 동안 개발 val 은 0.892 → 0.892 였다.")
print("   판정은 held-out(NightOwls rec 34) — 로컬에서 eval_nightowls.py 로.")
print("=" * 70)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1886.4±645.3 MB/s, size: 81.6 KB)
val: Scanning /content/work/datasets/aihub_colab_rehearsal/labels/val.cache... 45 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 45/45 18.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6it/s 1.8s0.9s
                   all         45        130      0.802      0.444      0.563      0.341
                person         41         93      0.837      0.591      0.715      0.459
               bollard         16         37      0.768      0.297       0.41      0.223
Speed: 7.3ms preprocess, 14.6ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to /content/drive/MyDrive/bammasil/runs/c4c_aihub_colab_val
class          mAP50    mAP50-95   precis

---
## 8. 산출물 정리

가중치는 이미 Drive 에 있다(6번 셀에서 직접 썼다). 여기서는 **재현에 필요한 것을
같이 묶는다** — 3주 뒤에 "이 가중치가 뭐로 학습된 거지"를 못 답하면 못 쓰는 가중치다.

In [27]:
import datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
src_run = PROJECT / RUN_NAME

meta = {
    "run": RUN_NAME,
    "생성": stamp,
    "노트북": "notebooks/colab_aihub_train.ipynb",
    "클래스": {str(k): v for k, v in CLASS_NAMES.items()},
    "데이터": {
        "zip": str(DATASET_ZIP),
        "이미지수": counts,
        "국내_패키징_stats": stats,
    },
    "학습": {"model": "yolo11n.pt", "epochs": EPOCHS, "imgsz": 640, "batch": BATCH,
             "seed": 0, "patience": 20, "resume": RESUME},
    "환경": {"ultralytics": ultralytics.__version__, "torch": torch.__version__,
             "gpu": gpu},
    "개발val_지표": per_class,
    "주의": "주간 전용 데이터. 야간 성능은 여기서 판정되지 않는다 (held-out/자체촬영에서).",
}
(src_run / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2),
                                   encoding="utf-8")

print("Drive 산출물:", src_run)
!ls -lh {src_run} {src_run}/weights

Drive 산출물: /content/drive/MyDrive/bammasil/runs/c4c_aihub_colab
/content/drive/MyDrive/bammasil/runs/c4c_aihub_colab:
total 5.5M
-rw------- 1 root root 1.8K Aug  4 08:42 args.yaml
-rw------- 1 root root 143K Aug  4 08:48 BoxF1_curve.png
-rw------- 1 root root 126K Aug  4 08:48 BoxP_curve.png
-rw------- 1 root root 109K Aug  4 08:48 BoxPR_curve.png
-rw------- 1 root root 131K Aug  4 08:48 BoxR_curve.png
-rw------- 1 root root 128K Aug  4 08:48 confusion_matrix_normalized.png
-rw------- 1 root root 124K Aug  4 08:48 confusion_matrix.png
-rw------- 1 root root 116K Aug  4 08:42 labels.jpg
-rw------- 1 root root 2.1K Aug  4 08:48 meta.json
-rw------- 1 root root  12K Aug  4 08:48 results.csv
-rw------- 1 root root 283K Aug  4 08:48 results.png
-rw------- 1 root root 644K Aug  4 08:42 train_batch0.jpg
-rw------- 1 root root 634K Aug  4 08:42 train_batch1.jpg
-rw------- 1 root root 650K Aug  4 08:42 train_batch2.jpg
-rw------- 1 root root 479K Aug  4 08:48 train_batch540.jpg
-rw------- 1 roo

---
## 9. 다음 — 로컬로 가져가서 할 것

Colab 은 **학습기**지 판정기가 아니다. 판정과 배포는 로컬에서 한다.

```powershell
# 1) Drive 에서 best.pt 를 받아 outputs/detect/c4c_aihub/weights/ 에 둔다

# 2) ★ 판정 — C4c 의 게이트는 "person recall 0.609 회귀 없음"이다.
#    볼라드를 얻자고 사람 성능을 깎으면 실패다
uv run python scripts/eval_nightowls.py --weights outputs/detect/c4c_aihub/weights/best.pt

# 3) 계단 야간 오탐이 되살아나지 않았는지 (C4b 가 0.2% 로 만들어 둔 것)
uv run python scripts/eval_stairs_night.py --weights outputs/detect/c4c_aihub/weights/best.pt

# 4) 통과하면 앱 전달물 재생성 (ONNX FP32 · 640 고정 · NMS 제외)
uv run python scripts/export_onnx.py --weights outputs/detect/c4c_aihub/weights/best.pt
```

### 문서에 남길 것

이 경로를 한 번 돌면 **문서의 "미확인" 항목들이 실측으로 바뀐다.**
`docs/STATUS.md` 를 먼저 갱신하고 상세는 `docs/data.md` 3-1-2 에 남길 것.

1. 노면 라벨 포맷 — CVAT XML polygon 이었나 PNG 마스크였나 (패키징 스크립트 출력)
2. 노면 `stairs` 규모와 크기 → 학습 투입인가 평가 전용인가 (게이트 결과)
3. **`bollard` recall 이 실제로 얼마인가** — 낮을 것이라 예상만 했지 잰 적이 없다
4. 노면 원천이 장애물 원천과 같은 이미지인가 (한 프레임에서 3클래스를 다 얻는가)